# Inicialização e organização dos rótulos


In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('/content/drive/MyDrive/lagenda/lagenda_annotation.csv')

idades_por_imagem = (
    df.groupby('img_name')['age']
      .apply(lambda x: list(x) if (x != -1).all() else None)
      .dropna()
      .reset_index()
)
# tem uma imagem faltando nos arquivos
idades_por_imagem = idades_por_imagem[idades_por_imagem['img_name'] != 'lag_benchmark/b5c8ef090f134ad5.jpg.jpg'].reset_index()

print(f"{len(idades_por_imagem)} imagens têm todas as idades conhecidas.")
display(idades_por_imagem)

20182 imagens têm todas as idades conhecidas.


,index,img_name,age
0,0,lag_benchmark/0000339d0372e7e6.jpg,[27]
1,1,lag_benchmark/0004350a376865f5.jpg,"[5, 6]"
2,2,lag_benchmark/00059fa95ee95e65.jpg,[79]
3,3,lag_benchmark/0007473cdbe8c18d.jpg,[6]
4,4,lag_benchmark/000851174d48d3d9.jpg,[4]
...,...,...,...
20177,20178,lag_benchmark/fff74322fa07c7c1.jpg,[41]
20178,20179,lag_benchmark/fff934bd1ff95af5.jpg,[76]
20179,20180,lag_benchmark/fffb40d02f510b35.jpg,[18]
20180,20181,lag_benchmark/ffff1ad4bf685255.jpg,[22]


In [ ]:
real_prompt1 = []
for idade in idades_por_imagem['age']:
  jovem = min(idade)
  if jovem < 14: #### menor que 14 :D
     valor = "1"
  else:
    valor = "0"
  real_prompt1 += [valor]
print(len(real_prompt1))

20182


In [ ]:
real_prompt2 = []
for idade in idades_por_imagem['age']:
  jovem = min(idade)
  if jovem < 18:
     valor = "1"
  else:
    valor = "0"
  real_prompt2 += [valor]

In [ ]:
real_prompt3 = []
for idade in idades_por_imagem['age']:
  jovem = min(idade)
  if jovem <= 13:
     valor = "criança"
  elif jovem <= 17:
    valor = "adolescente"
  elif jovem <= 59:
    valor = "adulto"
  else:
    valor = "idoso"
  real_prompt3 += [valor]

In [ ]:
real_prompt4 = []
for idade in idades_por_imagem['age']:
  jovem = min(idade)
  real_prompt4 += [jovem]

In [ ]:
real = [real_prompt1, real_prompt2, real_prompt3, real_prompt4]

# Construção da Tabela de resultados


In [ ]:
import glob
import os

In [ ]:
resultadosLlava = pd.DataFrame()

In [ ]:
#pasta onde estão os csvs
PASTA_CSV = "/content/drive/MyDrive/lagenda/grupo1"

# pega todos os arquivos csv
arquivos = sorted(
    glob.glob(os.path.join(PASTA_CSV, "*.csv")),
    key=lambda x: int(x.split("_p")[-1].split(".")[0])
)
print(arquivos)

# dataframe final
resultadosLlava['img_name'] = idades_por_imagem['img_name']

# percorre cada csv
for i, arquivo in enumerate(arquivos, start=1):

    # lê csv
    df_result = pd.read_csv(arquivo)

    nome_prev = f"previsao_prompt{i}"
    nome_real = f"real_prompt{i}"

    resultadosLlava[nome_prev] = df_result["resposta_modelo"]
    resultadosLlava[nome_real] = real[i-1]

resultadosLlava

['/content/drive/MyDrive/lagenda/grupo1/resultados_llava1.5_p1.csv', '/content/drive/MyDrive/lagenda/grupo1/resultados_llava1.5_p2.csv', '/content/drive/MyDrive/lagenda/grupo1/resultados_llava1.5_p3.csv', '/content/drive/MyDrive/lagenda/grupo1/resultados_llava1.5_p4.csv']


,img_name,previsao_prompt1,real_prompt1,previsao_prompt2,real_prompt2,previsao_prompt3,real_prompt3,previsao_prompt4,real_prompt4
0,lag_benchmark/0000339d0372e7e6.jpg,0,0,0,0,adolescente,adulto,18,27
1,lag_benchmark/0004350a376865f5.jpg,1,1,1,1,criança,criança,5,5
2,lag_benchmark/00059fa95ee95e65.jpg,0,0,0,0,adulto,idoso,10,79
3,lag_benchmark/0007473cdbe8c18d.jpg,1,1,1,1,criança,criança,5,6
4,lag_benchmark/000851174d48d3d9.jpg,1,1,1,1,criança,criança,5,4
...,...,...,...,...,...,...,...,...,...
20177,lag_benchmark/fff74322fa07c7c1.jpg,0,0,0,0,adolescente,adulto,25,41
20178,lag_benchmark/fff934bd1ff95af5.jpg,0,0,0,0,adulto,idoso,50,76
20179,lag_benchmark/fffb40d02f510b35.jpg,1,0,1,0,adolescente,adulto,16,18
20180,lag_benchmark/ffff1ad4bf685255.jpg,0,0,0,0,adolescente,adulto,18,22


In [ ]:
print(resultadosLlava['previsao_prompt1'].value_counts())

previsao_prompt1
0    12936
1     7246
Name: count, dtype: int64


**Tratamento de resultados do modelo**

In [ ]:
resultadosLlava['previsao_prompt1'] = resultadosLlava['previsao_prompt1'].apply(lambda x: '1' if x not in ['0', '1'] else x)

In [ ]:
resultadosLlava['previsao_prompt2'] = resultadosLlava['previsao_prompt2'].astype(str)

In [ ]:
resultadosLlava['previsao_prompt3'] = resultadosLlava['previsao_prompt3'].apply(lambda x: 'criança' if x == 'criança (0' else x)

In [ ]:
resultadosLlava['previsao_prompt3'] = resultadosLlava['previsao_prompt3'].apply(lambda x: 'adulto' if x == 'adulta' else x)

In [ ]:
resultadosLlava

,img_name,previsao_prompt1,real_prompt1,previsao_prompt2,real_prompt2,previsao_prompt3,real_prompt3,previsao_prompt4,real_prompt4
0,lag_benchmark/0000339d0372e7e6.jpg,0,0,0,0,adolescente,adulto,18,27
1,lag_benchmark/0004350a376865f5.jpg,1,1,1,1,criança,criança,5,5
2,lag_benchmark/00059fa95ee95e65.jpg,0,0,0,0,adulto,idoso,10,79
3,lag_benchmark/0007473cdbe8c18d.jpg,1,1,1,1,criança,criança,5,6
4,lag_benchmark/000851174d48d3d9.jpg,1,1,1,1,criança,criança,5,4
...,...,...,...,...,...,...,...,...,...
20177,lag_benchmark/fff74322fa07c7c1.jpg,0,0,0,0,adolescente,adulto,25,41
20178,lag_benchmark/fff934bd1ff95af5.jpg,0,0,0,0,adulto,idoso,50,76
20179,lag_benchmark/fffb40d02f510b35.jpg,1,0,1,0,adolescente,adulto,16,18
20180,lag_benchmark/ffff1ad4bf685255.jpg,0,0,0,0,adolescente,adulto,18,22


###Análise de Dados

In [19]:
from sklearn.metrics import (balanced_accuracy_score, precision_score,recall_score, mean_absolute_error, accuracy_score)

In [ ]:
rotulo = resultadosLlava['real_prompt1']
previsao = resultadosLlava['previsao_prompt1']
round(balanced_accuracy_score(rotulo, previsao)*100, 1), round(precision_score(rotulo, previsao, pos_label='1')*100, 1), round(recall_score(rotulo, previsao, pos_label='1')*100, 1)

(np.float64(91.0), 65.9, 98.1)

**Multiclasses**

In [ ]:
rotulo = resultadosLlava['real_prompt3']
previsao = resultadosLlava['previsao_prompt3']
round(balanced_accuracy_score(rotulo, previsao)*100, 1), round(precision_score(rotulo, previsao, average='weighted')*100, 1), round(recall_score(rotulo, previsao, average='weighted')*100, 1)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(np.float64(49.8), 70.2, 37.7)

**Idade**

In [ ]:
for clas in ['criança', 'adolescente', 'adulto', 'idoso']:
    rotulo = resultadosLlava[resultadosLlava['real_prompt3'] == clas]['real_prompt4']
    previsao = resultadosLlava[resultadosLlava['real_prompt3'] == clas]['previsao_prompt4']

    print(f'MAE {clas}:', round(mean_absolute_error(rotulo, previsao), 1))

MAE criança: 5.2
MAE adolescente: 6.1
MAE adulto: 18.0
MAE idoso: 42.4


# No-Face

## Construção da Tabela de resultados

In [ ]:
resultadosLlava_nf = pd.DataFrame()

In [ ]:
# pasta onde estão os csvs
PASTA_CSV = "/content/drive/MyDrive/lagenda/grupo1-noface"

# pega todos os arquivos csv
arquivos = sorted(
    glob.glob(os.path.join(PASTA_CSV, "*.csv")),
    key=lambda x: int(x.split("_p")[-1].split(".")[0])
)
print(arquivos)

# dataframe final
resultadosLlava_nf['img_name'] = idades_por_imagem['img_name']

# percorre cada csv
for i, arquivo in enumerate(arquivos, start=1):

    # lê csv
    df_result = pd.read_csv(arquivo)

    nome_prev = f"previsao_prompt{i}"
    nome_real = f"real_prompt{i}"

    resultadosLlava_nf[nome_prev] = df_result["resposta_modelo"]
    resultadosLlava_nf[nome_real] = real[i-1]

resultadosLlava_nf

['/content/drive/MyDrive/lagenda/grupo1-noface/resultados_llava1.5_p1.csv', '/content/drive/MyDrive/lagenda/grupo1-noface/resultados_llava1.5_p2.csv', '/content/drive/MyDrive/lagenda/grupo1-noface/resultados_llava1.5_p3.csv', '/content/drive/MyDrive/lagenda/grupo1-noface/resultados_llava1.5_p4.csv']


,img_name,previsao_prompt1,real_prompt1,previsao_prompt2,real_prompt2,previsao_prompt3,real_prompt3,previsao_prompt4,real_prompt4
0,lag_benchmark/0000339d0372e7e6.jpg,0,0,0,0,adolescente,adulto,18,27
1,lag_benchmark/0004350a376865f5.jpg,1,1,1,1,criança,criança,5,5
2,lag_benchmark/00059fa95ee95e65.jpg,0,0,0,0,adolescente,idoso,10,79
3,lag_benchmark/0007473cdbe8c18d.jpg,1,1,1,1,criança,criança,10,6
4,lag_benchmark/000851174d48d3d9.jpg,1,1,1,1,criança,criança,10,4
...,...,...,...,...,...,...,...,...,...
20177,lag_benchmark/fff74322fa07c7c1.jpg,0,0,0,0,adolescente,adulto,20,41
20178,lag_benchmark/fff934bd1ff95af5.jpg,0,0,0,0,adulto,idoso,20,76
20179,lag_benchmark/fffb40d02f510b35.jpg,1,0,1,0,adolescente,adulto,16,18
20180,lag_benchmark/ffff1ad4bf685255.jpg,0,0,0,0,adolescente,adulto,10,22


In [ ]:
print(resultadosLlava_nf['previsao_prompt3'].value_counts())


previsao_prompt3
adolescente             10934
adulto                   4621
criança                  4617
idoso                       4
bertye lou woods            1
miss bee-hayvin'            1
miss kimmberly walts        1
jennifer radloff            1
barbara j. patane           1
j. beaty jr.                1
Name: count, dtype: int64


**Tratamento de resultados do modelo**

In [ ]:
resultadosLlava_nf['previsao_prompt1'] = resultadosLlava_nf['previsao_prompt1'].apply(lambda x: '1' if x not in ['0', '1'] else x).astype(str)

In [ ]:
resultadosLlava_nf['previsao_prompt2'] = resultadosLlava_nf['previsao_prompt2'].apply(lambda x: '1' if x not in ['0', '1'] else x).astype(str)

In [ ]:
resultadosLlava_nf['previsao_prompt3'] = resultadosLlava_nf['previsao_prompt3'].apply(lambda x: 'criança' if x == 'criança (0-12)' else x)

In [ ]:
resultadosLlava_nf['previsao_prompt3'] = resultadosLlava_nf['previsao_prompt3'].apply(lambda x: 'criança' if x == 'criança (0–12)' else x)

In [ ]:
resultadosLlava_nf['previsao_prompt3'] = resultadosLlava_nf['previsao_prompt3'].apply(lambda x: 'adulto' if x == 'adulta' else x)

### Análise de Dados

In [ ]:
rotulo = resultadosLlava_nf['real_prompt1']
previsao = resultadosLlava_nf['previsao_prompt1']
round(balanced_accuracy_score(rotulo, previsao)*100, 1), round(precision_score(rotulo, previsao, pos_label='1')*100, 1), round(recall_score(rotulo, previsao, pos_label='1')*100, 1)

(np.float64(88.5), 69.1, 89.8)

In [ ]:
rotulo = resultadosLlava_nf['real_prompt3']
previsao = resultadosLlava_nf['previsao_prompt3']
round(balanced_accuracy_score(rotulo, previsao)*100, 1), round(precision_score(rotulo, previsao, average='weighted')*100, 1), round(recall_score(rotulo, previsao, average='weighted')*100, 1)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(np.float64(46.5), 66.3, 32.5)

In [ ]:
for clas in ['criança', 'adolescente', 'adulto', 'idoso']:
    rotulo = resultadosLlava_nf[resultadosLlava_nf['real_prompt3'] == clas]['real_prompt4']
    previsao = resultadosLlava_nf[resultadosLlava_nf['real_prompt3'] == clas]['previsao_prompt4']

    print(f'MAE {clas}:', round(mean_absolute_error(rotulo, previsao), 1))

MAE criança: 6.8
MAE adolescente: 6.4
MAE adulto: 19.5
MAE idoso: 48.8


## JSON Prompts

### organização dos rótulos

In [15]:
# idades_por_imagem

real = {'real_prompt1': [], 'real_prompt2': [],
        'real_prompt3': [], 'real_prompt4': []}

for idade in idades_por_imagem['age']:
    real['real_prompt1'].append([])
    real['real_prompt2'].append([])
    real['real_prompt3'].append([])
    real['real_prompt4'].append([])

    for i in idade:
        real['real_prompt1'][-1].append("1" if i <= 13 else "0")
        real['real_prompt2'][-1].append("1" if i <= 17 else "0")
        real['real_prompt4'][-1].append(i)

        if i <= 13:
            valor = "criança"
        elif i <= 17:
            valor = "adolescente"
        elif i <= 59:
            valor = "adulto"
        else:
            valor = "idoso"

        real['real_prompt3'][-1].append(valor)

In [16]:
#Sem oclusão da face
pred_prompt1 = pd.read_csv('/content/drive/MyDrive/lagenda/grupo2/resultados_llava1.5_json_p1.csv')
pred_prompt2 = pd.read_csv('/content/drive/MyDrive/lagenda/grupo2/resultados_llava1.5_json_p2.csv')
pred_prompt3 = pd.read_csv('/content/drive/MyDrive/lagenda/grupo2/resultados_llava1.5_json_p3.csv')
pred_prompt4 = pd.read_csv('/content/drive/MyDrive/lagenda/grupo2/resultados_llava1.5_json_p4.csv')

In [ ]:
#Com oclusão da face
exp = 'noface'
pred_prompt1 = pd.read_csv(f'json-prompt1-{exp}/resultados_llava1.5_json_prompt1_noface.csv').sort_values('img_name')
pred_prompt2 = pd.read_csv(f'json-prompt2-{exp}/resultados_llava1.5_json_prompt2-noface.csv').sort_values('img_name')
pred_prompt3 = pd.read_csv(f'json-prompt3-{exp}/resultados_llava1.5_json_prompt3_noface.csv').sort_values('img_name')
pred_prompt4 = pd.read_csv(f'json-prompt4-{exp}/resultados_llava1.5_json_prompt4_noface.csv').sort_values('img_name')

In [17]:
resultadosLlavaJSON = pd.DataFrame()

resultadosLlavaJSON['img_name'] = idades_por_imagem['img_name']
resultadosLlavaJSON['real_prompt1'] = real['real_prompt1']
resultadosLlavaJSON['real_prompt2'] = real['real_prompt2']

for nome, pred in zip(['pred_prompt1', 'pred_prompt2'], [pred_prompt1, pred_prompt2]):
    respostas = []
    for indice in range(len(pred)):
        linha = pred.iloc[indice]
        respostas.append([])

        try:
            dict_resultado = eval(linha['resposta_modelo'])
            for pessoa in dict_resultado['pessoas']:
                respostas[-1].append("1" if pessoa['categoria'] == 'criança' else "0")
        except:
            txt = linha['resposta_modelo']
            respostas[-1] = ["1"] * txt.count('criança')
            respostas[-1].extend(["0"] * txt.count('adulto'))

    resultadosLlavaJSON[nome] = respostas

##################
str2class = lambda lst: [['criança', 'adolescente', 'adulto', 'idoso'].index(item) for item in lst]
resultadosLlavaJSON['real_prompt3'] = list(map(str2class, real['real_prompt3']))

respostas = []
for indice in range(len(pred_prompt3)):
    linha = pred_prompt3.iloc[indice]
    respostas.append([])

    try:
        dict_resultado = eval(linha['resposta_modelo'])
        for pessoa in dict_resultado['pessoas']:
            if pessoa['categoria'] == 'adulta':
                pessoa['categoria'] = 'adulto'
            respostas[-1].append(['criança', 'adolescente', 'adulto', 'idoso'].index(pessoa['categoria'].lower()))
    except:
        txt = linha['resposta_modelo'].lower()
        txt = ' '.join(txt.split('}')[:-1])
        respostas[-1].extend([0] * txt.count('criança'))
        respostas[-1].extend([1] * txt.count('adolescente'))
        respostas[-1].extend([2] * txt.count('adulto'))
        respostas[-1].extend([3] * txt.count('idoso'))

resultadosLlavaJSON['pred_prompt3'] = respostas


resultadosLlavaJSON['real_prompt4'] = real['real_prompt4']
respostas = []
for indice in range(len(pred_prompt4)):
    linha = pred_prompt4.iloc[indice]
    respostas.append([])

    try:
        dict_resultado = eval(linha['resposta_modelo'])
        for pessoa in dict_resultado['pessoas']:
            respostas[-1].append(pessoa['idade'])
    except:
        txt = linha['resposta_modelo']
        pos = txt.find('idade')
        while pos != -1:
            respostas[-1].append(txt[pos:].split(':')[1].split('}')[0].strip())
            pos = txt.find('idade', pos+1)

resultadosLlavaJSON['pred_prompt4'] = respostas
resultadosLlavaJSON['pred_prompt4'] = resultadosLlavaJSON['pred_prompt4'].apply(lambda lst: [min(int(x), 100) for x in lst])

In [20]:
from sklearn.preprocessing import MultiLabelBinarizer

#1, 2, 3
for i in range(1, 4):
    mlb = MultiLabelBinarizer()
    y_true_bin = mlb.fit_transform(resultadosLlavaJSON[f'real_prompt{i}'])
    y_pred_bin = mlb.transform(resultadosLlavaJSON[f'pred_prompt{i}'])

    acc = accuracy_score(y_true_bin, y_pred_bin,)
    prec = precision_score(y_true_bin, y_pred_bin, average='weighted')
    rec = recall_score(y_true_bin, y_pred_bin, average='weighted')
    print(acc, prec, rec)

0.7909523337627589 0.8613816309541659 0.9593420083476553
0.7723714200772966 0.8445479624843909 0.9343015553701978
0.6863541769893965 0.7149361229186445 0.7626570232204035


In [21]:
from scipy.stats import wasserstein_distance
import numpy as np

for clas in range(4):
    val = []
    rotulo = resultadosLlavaJSON[resultadosLlavaJSON['real_prompt3'].apply(lambda x: clas in x and len(x) > 1)]['real_prompt4']
    previsao = resultadosLlavaJSON[resultadosLlavaJSON['real_prompt3'].apply(lambda x: clas in x and len(x) > 1)]['pred_prompt4']

    for gt, y in zip(rotulo, previsao):
        w = wasserstein_distance(gt, y)
        val.append(w)
    print(np.mean(val))

4.154268916155419
4.265789473684211
8.262609786700125
10.077616161616161


In [22]:
# set overlap coefficient
overlap_coefficient = lambda set1,set2: len(set1 & set2) / max(len(set1), len(set2))

for clas in range(4):
    rotulo = resultadosLlavaJSON[resultadosLlavaJSON['real_prompt3'].apply(lambda x: clas in x and len(x) > 1)]['real_prompt2']
    previsao = resultadosLlavaJSON[resultadosLlavaJSON['real_prompt3'].apply(lambda x: clas in x and len(x) > 1)]['pred_prompt2']

    cont = len(rotulo)
    soma = 0
    for gt, y in zip(rotulo, previsao):
        soma += overlap_coefficient(set(gt), set(y))
    print(f'Mean Overlap {clas}:', soma/cont)

Mean Overlap 0: 0.6196319018404908
Mean Overlap 1: 0.7789473684210526
Mean Overlap 2: 0.5683814303638645
Mean Overlap 3: 0.5393939393939394
